In [1]:
# Instalar dependências (execute esta célula uma vez)
!pip install -q pdfplumber deep-translator transformers torch accelerate

# Imports
import pdfplumber
from deep_translator import GoogleTranslator
from transformers import pipeline
import torch

# Função para dividir texto em chunks (aproximando sentenças)
def split_into_chunks(text, max_size):
    chunks = []
    sentences = text.split('.')
    current = ''
    for sent in sentences:
        sent = sent.strip() + '.'
        if len((current + sent).strip()) <= max_size:
            current += sent + ' '
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sent + ' '
    if current.strip():
        chunks.append(current.strip())
    return [c for c in chunks if c]  # Remove vazios

# Caminho do PDF
pdf_path = input("Digite o caminho completo para o arquivo PDF: ")

# 1. Extrair texto do PDF
print("Extraindo texto do PDF...")
with pdfplumber.open(pdf_path) as pdf:
    text = '\n'.join([page.extract_text() or '' for page in pdf.pages])
print(f'Texto extraído: {len(text)} caracteres')

# 2. Dividir em chunks para tradução
chunks = split_into_chunks(text, 4500)
print(f'Dividido em {len(chunks)} chunks para tradução')

# 3. Traduzir cada chunk (Google Translate gratuito)
translator = GoogleTranslator(source='en', target='pt')
print('Traduzindo...')
translated_chunks = []
for i, chunk in enumerate(chunks):
    print(f'Traduzindo chunk {i+1}/{len(chunks)}')
    try:
        trans = translator.translate(chunk)
        translated_chunks.append(trans)
    except Exception as e:
        print(f'Erro na tradução do chunk {i+1}: {e}')
        translated_chunks.append(chunk)  # Mantém original em caso de erro
full_text_pt = '\n'.join(translated_chunks)
print(f'Texto traduzido: {len(full_text_pt)} caracteres')

# 4. Carregar modelo de sumarização (Flan-T5 multilingual, excelente para resumos em PT)
print('Carregando modelo de sumarização...')
model_name = 'google/flan-t5-base'  # Rápido e funcional; use 'google/flan-t5-large' para melhor qualidade
device = 0 if torch.cuda.is_available() else -1
summarizer = pipeline(
    'text2text-generation',
    model=model_name,
    device=device,
    torch_dtype=torch.float16 if device == 0 else torch.float32
)

# 5. Função para resumir texto longo (map-reduce: resume chunks, depois resume o combinado)
def summarize_long_text(text, chunk_size=1200, max_new_tokens=250, min_new_tokens=50):
    chunks = split_into_chunks(text, chunk_size)
    chunk_summaries = []
    for chunk in chunks:
        prompt = f'Resuma este texto em português:\n\n{chunk}'
        summary = summarizer(prompt, max_new_tokens=max_new_tokens, min_new_tokens=min_new_tokens, do_sample=False)[0]['generated_text']
        chunk_summaries.append(summary)
    combined = ' '.join(chunk_summaries)
    # Resumo final mais conciso
    final_prompt = f'Resuma de forma concisa em português:\n\n{combined}'
    final_summary = summarizer(final_prompt, max_new_tokens=350, min_new_tokens=100, do_sample=False)[0]['generated_text']
    return final_summary

# 6. Gerar resumo em português
print('Gerando resumo em português...')
resumo = summarize_long_text(full_text_pt)

# 7. Exibir resultado
print('\n' + '='*60)
print('RESUMO EM PORTUGUÊS:')
print(resumo)
print('='*60)

# Salvar em arquivo .txt
output_path = 'resumo_portugues.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(resumo)
print(f'\nResumo salvo em: {output_path}')

print('\nProcesso concluído com sucesso!')

KeyboardInterrupt: Interrupted by user

In [3]:
import os
os.environ["MPLBACKEND"] = "agg"
import pdfplumber
from transformers import pipeline
from deep_translator import GoogleTranslator
import torch
import os

# 1. Função para dividir texto em blocos (chunks)
def split_into_chunks(text, max_size=2500):
    chunks = []
    sentences = text.split('. ')
    current = ''
    for sent in sentences:
        if len(current) + len(sent) < max_size:
            current += sent + '. '
        else:
            if current: chunks.append(current.strip())
            current = sent + '. '
    if current: chunks.append(current.strip())
    return [c for c in chunks if c]

# 2. Função para traduzir textos de qualquer tamanho (evita erro NotValidLength)
def translate_long_text(text, src='en', tgt='pt'):
    translator = GoogleTranslator(source=src, target=tgt)
    # Divide em blocos de 4000 caracteres (limite do Google é 5000)
    parts = split_into_chunks(text, 4000)
    translated_parts = []
    
    total = len(parts)
    for i, part in enumerate(parts):
        if total > 1:
            print(f"      Sub-bloco de tradução {i+1}/{total}...")
        try:
            translated_parts.append(translator.translate(part))
        except Exception as e:
            print(f"      Erro no sub-bloco {i+1}: {e}")
            translated_parts.append(part)
            
    return " ".join(translated_parts)

def process_turing_pdf(pdf_path):
    # --- Passo A: Extração ---
    print("--- 1. Extraindo texto do PDF ---")
    with pdfplumber.open(pdf_path) as pdf:
        text_en = "\n".join([page.extract_text() or '' for page in pdf.pages])

    # --- Passo B: Sumarização em Inglês ---
    print("--- 2. Gerando resumo em Inglês ---")
    device = 0 if torch.cuda.is_available() else -1
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=device)
    
    en_chunks = split_into_chunks(text_en, 3000)
    partial_summaries = []

    for i, chunk in enumerate(en_chunks):
        print(f"   Processando resumo do bloco {i+1}/{len(en_chunks)}...")
        summary = summarizer(
            chunk, 
            max_length=150, 
            min_length=50, 
            do_sample=False,
            repetition_penalty=2.5,
            no_repeat_ngram_size=3
        )[0]['summary_text']
        partial_summaries.append(summary)

    combined_summary_en = " ".join(partial_summaries)

    # --- Passo C: Traduções ---
    print("--- 3. Traduzindo Resumo Final ---")
    resumo_pt = translate_long_text(combined_summary_en)
    
    print("--- 4. Traduzindo Texto Completo (isso pode levar alguns minutos) ---")
    texto_completo_pt = translate_long_text(text_en)
    
    return resumo_pt, texto_completo_pt

# --- Execução ---
pdf_path = r"D:\TreinaRecife\Python do Zero até a Análise de Dados\aprendizado\Códigos\turing.pdf"

if os.path.exists(pdf_path):
    try:
        resumo, texto_completo = process_turing_pdf(pdf_path)
        
        # Salvar Resumo
        with open("resumo_final_pt.txt", "w", encoding="utf-8") as f:
            f.write(resumo)
            
        # Salvar Texto Completo
        with open("texto_completo_traduzido_pt.txt", "w", encoding="utf-8") as f:
            f.write(texto_completo)
            
        print("\n" + "="*60)
        print("SUCESSO!")
        print(f"Resumo salvo em: resumo_final_pt.txt ({len(resumo)} caracteres)")
        print(f"Texto completo salvo em: texto_completo_traduzido_pt.txt ({len(texto_completo)} caracteres)")
        print("="*60)
        
    except Exception as e:
        print(f"\nErro durante o processo: {e}")
else:
    print("Arquivo não encontrado no caminho especificado.")

--- 1. Extraindo texto do PDF ---
--- 2. Gerando resumo em Inglês ---


config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


   Processando resumo do bloco 1/24...
   Processando resumo do bloco 2/24...
   Processando resumo do bloco 3/24...
   Processando resumo do bloco 4/24...
   Processando resumo do bloco 5/24...
   Processando resumo do bloco 6/24...
   Processando resumo do bloco 7/24...
   Processando resumo do bloco 8/24...
   Processando resumo do bloco 9/24...
   Processando resumo do bloco 10/24...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


   Processando resumo do bloco 11/24...
   Processando resumo do bloco 12/24...
   Processando resumo do bloco 13/24...
   Processando resumo do bloco 14/24...
   Processando resumo do bloco 15/24...
   Processando resumo do bloco 16/24...
   Processando resumo do bloco 17/24...
   Processando resumo do bloco 18/24...
   Processando resumo do bloco 19/24...
   Processando resumo do bloco 20/24...
   Processando resumo do bloco 21/24...
   Processando resumo do bloco 22/24...
   Processando resumo do bloco 23/24...
   Processando resumo do bloco 24/24...
--- 3. Traduzindo Resumo Final ---
      Sub-bloco de tradução 1/3...
      Sub-bloco de tradução 2/3...
      Sub-bloco de tradução 3/3...
--- 4. Traduzindo Texto Completo (isso pode levar alguns minutos) ---
      Sub-bloco de tradução 1/18...
      Sub-bloco de tradução 2/18...
      Sub-bloco de tradução 3/18...
      Sub-bloco de tradução 4/18...
      Sub-bloco de tradução 5/18...
      Sub-bloco de tradução 6/18...
      Sub-bloc

In [4]:
resumo

'AM Turing (1950) Máquinas de Computação e Inteligência. Mente 49: 433-460.MÁQUINAS DE COMPUTAÇÃO E INTELIGÊNCIA \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 Por A. M.-Turing (1950) O Jogo da Imitação é um jogo jogado com três pessoas, um homem, uma mulher e um interrogador. O método de perguntas e respostas parece ser adequado para introduzir quase qualquer um dos campos da atividade humana que desejamos incluir. Nenhum engenheiro ou químico afirma ser capaz de produzir um material indistinguível da pele humana. As “testemunhas” podem gabar-se, se considerarem aconselhável, tanto quanto quiserem, dos seus encantos, força ou heroísmo, mas o interrogador não pode exigir demonstrações práticas. Se o homem tentasse fingir ser a máquina, teria um desempenho muito ruim. A questão que colocamos em 1 não será totalmente definida até que tenhamos especificado o que queremos dizer com a palavra “máquina”. É natural que desejássemos permitir que todo tipo de técnica de engenharia fosse usada em nossa

In [5]:
texto_completo

'AM Turing (1950) Máquinas de Computação e Inteligência. Mente 49: 433-460.\nMÁQUINAS DE COMPUTAÇÃO E INTELIGÊNCIA\nPor A. M. Turing\n1. O jogo da imitação\nProponho considerar a questão: “As máquinas podem pensar?” Isto deve começar com\ndefinições do significado dos termos "máquina" e "pensar". As definições podem ser\nenquadrado de modo a refletir, tanto quanto possível, o uso normal das palavras, mas esta atitude é\nperigoso, se o significado das palavras “máquina” e “pensar” for encontrado por\nexaminando como eles são comumente usados, é difícil escapar da conclusão de que o\nsignificado e a resposta à pergunta: "As máquinas podem pensar?" deve ser procurado em um\npesquisa estatística, como uma pesquisa Gallup. Mas isso é um absurdo. Em vez de tentar tal\ndefinição, substituirei a questão por outra, que está intimamente relacionada com ela e é\nexpresso em palavras relativamente inequívocas.\nA nova forma do problema pode ser descrita em termos de um jogo que chamamos de\n\'jogo